# Tutorial 04 — 30+ Models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/04_models.ipynb)

**Vectrix ships with 30+ forecasting models** spanning exponential smoothing, ARIMA, decomposition, theta methods, intermittent demand, volatility, neural reservoirs, and pattern matching.

This tutorial shows how to compare models, use the `Vectrix` class for full control, access individual engines directly, and understand the model selection process.

| What you'll learn | Time |
|:------------------|:-----|
| Quick model comparison | 1 min |
| The Vectrix class | 2 min |
| Direct engine access | 2 min |
| Flat prediction defense | 1 min |
| DNA-based model selection | 2 min |

In [ ]:
!pip install -q vectrix

## 1. Quick Model Comparison

One function call, all models ranked by accuracy.

In [ ]:
from vectrix import compare, loadSample

df = loadSample("airline")
ranking = compare(df, date="date", value="passengers", steps=12)
ranking

In [ ]:
# Also available from a forecast result
from vectrix import forecast

result = forecast(df, date="date", value="passengers", steps=12)
result.compare()

## 2. The Vectrix Class

The Easy API (`forecast()`) is great for quick results. When you need full control — access to all model results, flat risk diagnostics, ensemble weights — use `Vectrix` directly.

In [ ]:
from vectrix import Vectrix, loadSample

df = loadSample("airline")
vx = Vectrix()
result = vx.forecast(df, dateCol="date", valueCol="passengers", steps=12, trainRatio=0.8)

print(f"Best model: {result.bestModelName}")
print(f"Predictions: {result.predictions[:6].round(1)}")

In [ ]:
# Access all model results
for modelId, mr in result.allModelResults.items():
    print(f"{mr.modelName:>20s}: MAPE={mr.mape:.2f}%, RMSE={mr.rmse:.2f}")

## 3. Available Models

All models are evaluated automatically. The tables below show what's included.

### Exponential Smoothing
| Model | Best For |
|-------|----------|
| AutoETS | Stable patterns, trend, seasonality |

### ARIMA
| Model | Best For |
|-------|----------|
| AutoARIMA | Stationary and differenced series |

### Decomposition
| Model | Best For |
|-------|----------|
| MSTL / AutoMSTL | Multiple seasonal periods |

### Theta Methods
| Model | Best For |
|-------|----------|
| Theta | General purpose, M3 winner |
| DOT | Strongest single model |
| 4Theta | Multi-theta ensemble, M4 top-tier |

### Others
| Model | Best For |
|-------|----------|
| AutoCES | Nonlinear, complex dynamics |
| TBATS | Complex multi-seasonality |
| Croston / SBA / TSB | Sparse, intermittent demand |
| GARCH / EGARCH / GJR | Financial volatility |
| ESN | Nonlinear dynamics, ensemble diversity |
| DTSF | Pattern-matching, hourly data |

## 4. Direct Engine Access

Use individual model engines directly. Each follows the same `fit()` → `predict()` interface.

In [ ]:
from vectrix.engine.ets import AutoETS
from vectrix.engine.theta import OptimizedTheta
from vectrix.engine.dot import DynamicOptimizedTheta
from vectrix.engine.ces import AutoCES
import numpy as np

data = np.array([100, 120, 130, 115, 140, 160, 150, 170, 195, 185, 200, 215,
                 110, 125, 135, 120, 145, 165, 155, 175, 200, 190, 210, 225])

models = {
    "AutoETS": AutoETS(period=12),
    "Theta": OptimizedTheta(period=12),
    "DOT": DynamicOptimizedTheta(period=12),
    "AutoCES": AutoCES(period=12),
}

for name, model in models.items():
    model.fit(data)
    preds, lower, upper = model.predict(steps=6)
    print(f"{name:>10s}: {preds.round(1)}")

## 5. Flat Prediction Defense

A common failure mode: the model outputs the same value for every future step. Vectrix includes a unique 4-level defense system.

In [ ]:
from vectrix import Vectrix, loadSample

df = loadSample("airline")
vx = Vectrix()
result = vx.forecast(df, dateCol="date", valueCol="passengers", steps=12)

fr = result.flatRisk
print(f"Risk Level: {fr.riskLevel.name} ({fr.riskScore:.2f})")
print(f"Strategy:   {fr.recommendedStrategy}")

### The 4 Levels

| Level | Component | What It Does |
|-------|-----------|-------------|
| 1 | FlatRiskDiagnostic | Pre-assessment: estimates risk before fitting |
| 2 | AdaptiveModelSelector | Selection: biases away from flat-prone models |
| 3 | FlatPredictionDetector | Detection: checks if output is flat |
| 4 | FlatPredictionCorrector | Correction: automatically fixes flat predictions |

## 6. DNA-Based Model Selection

Vectrix uses 65+ statistical features to prioritize the most promising model candidates.

In [ ]:
from vectrix import forecast, loadSample

datasets = ["airline", "retail", "stock", "energy", "intermittent"]

for name in datasets:
    df = loadSample(name)
    cols = df.columns.tolist()
    date_col = cols[0]
    value_col = cols[1]
    result = forecast(df, date=date_col, value=value_col, steps=12)
    print(f"{name:>15s} -> {result.model:>15s}  MAPE={result.mape:.2f}%")

Different datasets get different models — Vectrix adapts automatically.

## 7. Complete Example

In [ ]:
import numpy as np
from vectrix import forecast, compare

np.random.seed(42)
t = np.arange(100)
data = 50 + 0.5 * t + 20 * np.sin(2 * np.pi * t / 12) + np.random.randn(100) * 5

ranking = compare(data, steps=12)
print("Top 5 models:")
print(ranking.head(5))

result = forecast(data, steps=12)
print(f"\nSelected: {result.model}")
print(f"MAPE: {result.mape:.2f}%")
print(f"Next 12: {result.predictions.round(1)}")

---

**Next:** [Tutorial 05 — Adaptive Intelligence](05_adaptive.ipynb)